# 01 — Data Ingestion and Exploratory Data Analysis

**Goal:** pull the Liu et al. (2024) acute-stroke MI dataset directly via MOABB, verify montage/channel integrity, and audit basic spectral content before any epoching or spatial filtering happens.

Reference: Liu H. et al., *Scientific Data* 11, 131 (2024). 50 patients, ZhenTec NT1 wireless system, 29 EEG + 2 EOG channels, 500 Hz, reference CPz, ground FPz.

> **Network note:** the first call below downloads data into `~/mne_data` (cached afterwards). This requires outbound internet access to MOABB's data host.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
import mne

from src.data_loader import get_dataset, load_subject_raw, N_EEG_CHANNELS, N_EOG_CHANNELS, SFREQ_HZ, REFERENCE_CHANNEL

## 1. Instantiate the dataset

In [ ]:
dataset = get_dataset()
print(f'Subjects available: {len(dataset.subject_list)}')
print(f'Subject IDs: {dataset.subject_list[:10]}...' if len(dataset.subject_list) > 10 else dataset.subject_list)

## 2. Inspect Subject 1 — verify channel counts, sampling rate, reference

In [ ]:
subject_data = load_subject_raw(dataset.subject_list[0])

# subject_data is {session: {run: raw}} — grab the first raw object
first_session = next(iter(subject_data.values()))
first_run = next(iter(first_session.values()))
raw = first_run

eeg_picks = mne.pick_types(raw.info, eeg=True, eog=False)
eog_picks = mne.pick_types(raw.info, eeg=False, eog=True)

print(f'EEG channels found: {len(eeg_picks)} (expected {N_EEG_CHANNELS})')
print(f'EOG channels found: {len(eog_picks)} (expected {N_EOG_CHANNELS})')
print(f'Sampling rate: {raw.info["sfreq"]} Hz (expected {SFREQ_HZ})')
print(f'Reference: {raw.info.get("custom_ref_applied")}, channel names include CPz: {"CPz" in raw.ch_names}')

assert len(eeg_picks) == N_EEG_CHANNELS, 'EEG channel count mismatch — see Section 8 Montage Mismatch note'
assert len(eog_picks) == N_EOG_CHANNELS, 'EOG channel count mismatch'

## 3. Electrode layout on the 10-10 scalp montage

In [ ]:
fig = raw.plot_sensors(show_names=True, kind='topomap')
plt.title('Liu2024 — 10-10 montage, Subject 1')
plt.show()

## 4. Welch PSD across C3, Cz, C4 (1–100 Hz)

Looking for: a clear 50 Hz line-noise peak (mains, sanity check on acquisition) and an alpha/mu peak (~8–12 Hz) at rest, most visible at C3/C4 before any task filtering.

In [ ]:
central_chs = [ch for ch in ('C3', 'Cz', 'C4') if ch in raw.ch_names]
raw_eeg = raw.copy().pick(central_chs)

spectrum = raw_eeg.compute_psd(method='welch', fmin=1, fmax=100, n_fft=1024)
fig, ax = plt.subplots(figsize=(9, 5))
spectrum.plot(axes=ax, show=False, average=False)
ax.axvline(50, color='red', linestyle='--', alpha=0.5, label='50 Hz line noise')
ax.axvspan(8, 12, color='orange', alpha=0.15, label='mu band (8-12 Hz)')
ax.legend()
ax.set_title('Welch PSD — C3 / Cz / C4, Subject 1')
plt.tight_layout()
plt.show()

## Next
Proceed to **`02_Signal_Processing_and_ERD.ipynb`** to epoch the MI window and look for contralateral event-related desynchronization.